# 3.4 — Blend / stacking ML + DL và submission cuối

Ứng viên: 5 model ML (3.1) + các model DL PLR (3.3). Mọi phương án được chấm bằng **CV lồng theo 5 fold chung**:
trọng số / meta-model học trên 4 fold, chấm AUC trên fold còn lại → AUC trung bình theo fold (không lạc quan như tối ưu trên toàn OOF).

| Phương án | Mô tả |
|---|---|
| `ml_only` | rank blend 5 model ML (tối ưu trọng số, shrink 0.7/0.3 như ML task 1.5) — mốc so sánh |
| `ml+dl_best` | rank blend 5 ML + `mlp_plr_sig3_reg_x3` |
| `ml+dl_all` | rank blend 5 ML + mọi model DL PLR |
| `stack_lr` | Logistic Regression trên logit xác suất OOF của 5 ML + mọi DL |

Chọn phương án có AUC trung bình theo fold cao nhất, rồi học lại trên toàn OOF và áp lên test.

**Input:** competition + `ccf-shared-code` + output 3.0 + output 3.1 + output 3.3 (phiên bản mới nhất). **Accelerator:** None.
**Output:** `submission_ensemble.csv` (phương án được chọn), `submission_ml_only.csv` (để đối chiếu LB), `3.ENSEMBLE/results/blend_comparison.csv`, `final_blend.json`.

In [ ]:
import sys
from pathlib import Path

# shared/ code: local repo or the Kaggle dataset that holds features.py + dl.py
cands = [Path("shared/features.py"), Path("../../shared/features.py"), *Path("/kaggle/input").rglob("features.py")]
feat = next(p for p in cands if p.is_file())
sys.path.insert(0, str(feat.parent.resolve()))
import features as F
print("shared code:", feat.parent)

In [ ]:
import time
import json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

T0 = time.time()
def log(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

tr, te, y, train_id, test_id = F.load()
fold = F.load_folds(train_id)          # from notebook 3.0 output (folds_5_seed9.csv)
assert len(tr) == F.N_TRAIN and len(te) == F.N_TEST and set(np.unique(fold)) == set(range(F.N_FOLDS))
log(f"train {tr.shape}, test {te.shape}, fraud rate {y.mean():.4%}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import StandardScaler

ML = ["lgb_nl3", "lgb_nl4", "lgb_nl8", "spline_lr", "xgb"]
DL_BEST = ["mlp_plr_sig3_reg_x3"]
DL_ALL = ["mlp_plr_sig3_reg_x3", "mlp_plr_sig1_reg", "mlp_plr_sig3", "mlp_plr_sig10_reg"]
NAMES = ML + DL_ALL

O, T = F.load_predictions(NAMES, train_id, test_id)      # xác suất OOF (train) và trung bình 5 fold (test)
R_oof = np.column_stack([F.rank01(O[:, i]) for i in range(len(NAMES))])
R_test = np.column_stack([F.rank01(T[:, i]) for i in range(len(NAMES))])
col = {n: i for i, n in enumerate(NAMES)}
for n in NAMES:
    print(f"{n:22s} OOF AUC={roc_auc_score(y, O[:, col[n]]):.5f}")

In [ ]:
# ---------- các phương án: fit(rows) -> hàm score(R_or_O_rows)
def rank_blend(models, shrink=0.3, iters=500):
    idx = [col[m] for m in models]
    def fit(rows):
        w, _ = F.tune_weights(R_oof[np.ix_(rows, idx)], y[rows], iters=iters)
        w = (1 - shrink) * w + shrink / len(idx)
        return {"weights": dict(zip(models, w.tolist()))}, lambda R, O_: R[:, idx] @ w
    return fit

def logit(p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return np.log(p / (1 - p))

def stack_lr(models, C=0.1):
    idx = [col[m] for m in models]
    def fit(rows):
        sc = StandardScaler().fit(logit(O[np.ix_(rows, idx)]))
        lr = LogisticRegression(C=C, max_iter=1000).fit(sc.transform(logit(O[np.ix_(rows, idx)])), y[rows])
        coef = dict(zip(models, lr.coef_[0].tolist()))
        return {"coef": coef}, lambda R, O_: lr.decision_function(sc.transform(logit(O_[:, idx])))
    return fit

METHODS = {
    "ml_only": rank_blend(ML),
    "ml+dl_best": rank_blend(ML + DL_BEST),
    "ml+dl_all": rank_blend(ML + DL_ALL),
    "stack_lr": stack_lr(ML + DL_ALL),
}

In [ ]:
# ---------- CV lồng: học trên 4 fold, chấm trên fold còn lại
rows_out = []
for name, fit in METHODS.items():
    fold_auc, fold_ap = [], []
    for f, a, b in F.iter_folds(fold):
        _, score = fit(a)
        s = score(R_oof[b], O[b])
        fold_auc.append(roc_auc_score(y[b], s))
        fold_ap.append(average_precision_score(y[b], s))
    rows_out.append({"method": name, "fold_auc_mean": np.mean(fold_auc), "fold_auc_std": np.std(fold_auc),
                     "fold_ap_mean": np.mean(fold_ap), "fold_auc": np.round(fold_auc, 5).tolist()})
    log(f"{name:12s} fold AUC mean={np.mean(fold_auc):.5f} ± {np.std(fold_auc):.5f} | AP={np.mean(fold_ap):.5f}")

cmp = pd.DataFrame(rows_out).set_index("method")
cmp["gain_vs_ml_only"] = cmp["fold_auc_mean"] - cmp.loc["ml_only", "fold_auc_mean"]
# số fold mà phương án thắng ml_only
base = np.array(cmp.loc["ml_only", "fold_auc"])
cmp["folds_beating_ml_only"] = [int((np.array(v) > base).sum()) for v in cmp["fold_auc"]]
pd.set_option("display.width", 200)
print(cmp.round(5))
Path("3.ENSEMBLE/results").mkdir(parents=True, exist_ok=True)
cmp.to_csv("3.ENSEMBLE/results/blend_comparison.csv")

In [ ]:
# ---------- chọn phương án: AUC trung bình theo fold cao nhất; chỉ bỏ ml_only nếu thắng ở ≥ 4/5 fold
best = cmp["fold_auc_mean"].idxmax()
if best != "ml_only" and cmp.loc[best, "folds_beating_ml_only"] < 4:
    print(f"{best} không thắng ml_only ổn định ({cmp.loc[best, 'folds_beating_ml_only']}/5 fold) -> giữ ml_only")
    best = "ml_only"
print("Phương án được chọn:", best)

all_rows = np.arange(len(y))
params, score = METHODS[best](all_rows)
test_score = score(R_test, T)
oof_score = score(R_oof, O)

ml_params, ml_score = METHODS["ml_only"](all_rows)
ml_test = ml_score(R_test, T)

final = {"selected_method": best, "selection": "max nested-CV fold-mean ROC-AUC on shared 5 folds (seed 9)",
         "nested_cv": cmp.loc[best, ["fold_auc_mean", "fold_auc_std", "fold_ap_mean", "gain_vs_ml_only"]].to_dict(),
         "fit_on_all_oof": params, "models": NAMES,
         "leaderboard_used_for_selection": False}
Path("3.ENSEMBLE/results/final_blend.json").write_text(json.dumps(final, indent=2, default=float))
print(json.dumps(final, indent=2, default=float))

In [ ]:
# ---------- submission: xếp theo sample_submission, đưa điểm về [0,1] bằng rank (AUC chỉ phụ thuộc thứ tự)
sample = pd.read_csv(F.find_file("sample_submission.csv"))
assert list(sample.columns) == ["id", "is_fraud"] and sample["id"].is_unique
assert set(sample["id"]) == set(test_id) and len(sample) == F.N_TEST

def write_submission(scores, path):
    s = pd.Series(F.rank01(scores), index=test_id)
    sub = pd.DataFrame({"id": sample["id"], "is_fraud": s.reindex(sample["id"]).to_numpy()})
    assert len(sub) == F.N_TEST and sub["is_fraud"].notna().all()
    assert np.isfinite(sub["is_fraud"]).all() and sub["is_fraud"].between(0, 1).all()
    sub.to_csv(path, index=False)
    print(f"saved {path}: {len(sub):,} rows, range [{sub['is_fraud'].min():.4f}, {sub['is_fraud'].max():.4f}]")
    return sub

sub = write_submission(test_score, "submission_ensemble.csv")
write_submission(ml_test, "submission_ml_only.csv")
print("Spearman(test) ensemble vs ml_only:", round(float(pd.Series(test_score).corr(pd.Series(ml_test), method="spearman")), 5))
print(sub.head())
log("done")